## BMLL Notebook Educational Series
### Closing Auction Imbalance & Price Formation

The closing auction is the most important event of the trading day: a large fraction
of daily volume executes at the official closing price, making it the primary benchmark
for passive funds, index products, and TCA benchmarks.

Exchanges publish a continuous feed of **auction imbalance** data throughout the closing
call period, revealing:
- The **indicative uncrossing price** — where the book would cross if the auction ended now
- The **paired quantity** — how much volume has been matched at the indicative price
- The **imbalance quantity** — the surplus of buy or sell orders that cannot be matched
- The **reference price** — the exchange's anchor price (often the last mid-price)

Tracking how these metrics evolve during the auction helps participants:
- Detect one-sided liquidity pressure and anticipate price direction
- Time participation orders to maximize fill probability
- Understand venue-specific auction dynamics

**Note:** This analysis covers venues that publish transparency data during the auction
(e.g., LSE/XLON, Euronext/XPAR, NYSE/XNYS, NASDAQ/XNAS). Not all venues publish
intraday imbalance feeds. For the price dislocation chart, three fallback tiers are
used: (1) exchange-published reference price, (2) auction BBO mid-price, (3) first
indicative uncrossing price of the day as an auction-open anchor.

In [ ]:
dbutils.widgets.text("mic", "XLON")
dbutils.widgets.text("ticker", "AZN")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")

mic = dbutils.widgets.get("mic")
ticker = dbutils.widgets.get("ticker")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

In [ ]:
# ── Table configuration ─────────────────────────────────────────────────
# Set USE_SAMPLE_DATA=true/false as an environment variable (or Databricks
# cluster/job env var) to switch between trial views (DD-1167) and full
# production views without changing any code.
# Override catalog names via MARKET_DATA_CATALOG / DERIVED_DATA_CATALOG.
import os as _os

_s = _os.environ.get("USE_SAMPLE_DATA", "").lower()
USE_SAMPLE_DATA = True if _s == "true" else (False if _s == "false" else False)

MARKET_DATA_CATALOG  = _os.environ.get("MARKET_DATA_CATALOG",  "market_data")
DERIVED_DATA_CATALOG = _os.environ.get("DERIVED_DATA_CATALOG", "derived_data")

if USE_SAMPLE_DATA:
    # DD-1167 trial views — 6-month rolling window, no subscription required
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.REF_GLOBAL_PROD"
    ANALYTICS_TBL               = f"{MARKET_DATA_CATALOG}.ANA_GLOBAL_PROD"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.TRDS_GLOBAL_PROD"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.TDSP_GLOBAL_PROD"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.L2_GLOBAL_PROD"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.CBBO_GLOBAL_PROD"
    DAILY_CLASSIFIED_TRADES_TBL = f"{MARKET_DATA_CATALOG}.DCT_GLOBAL_PROD"
    # No dedicated sample views for these tables — fall back to production views
    IMBALANCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_imbalance"
    MARKET_STATE_TBL            = f"{MARKET_DATA_CATALOG}.vw_market_state"
else:
    # Full production views
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_reference"
    ANALYTICS_TBL               = f"{DERIVED_DATA_CATALOG}.vw_analytics"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.vw_trades"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.vw_trades_plus"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.vw_l2_data_equity"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.vw_cbbo_equity"
    DAILY_CLASSIFIED_TRADES_TBL = f"{DERIVED_DATA_CATALOG}.vw_daily_classified_trades"
    IMBALANCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_imbalance"
    MARKET_STATE_TBL            = f"{MARKET_DATA_CATALOG}.vw_market_state"

In [ ]:
import datetime as _dt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# ── Rolling date-range guard ──────────────────────────────────────────────────
# The BMLL sample data covers a rolling 6-month window ending today.
# Reset any date range that falls entirely before that window so the notebook
# always runs correctly on both trial and production datasets.
_today = _dt.date.today()
_window_start = _today - _dt.timedelta(days=182)

if not start_date:
    start_date = (_today - _dt.timedelta(days=90)).isoformat()
if not end_date:
    end_date = _today.isoformat()

_sd = _dt.date.fromisoformat(start_date)
_ed = _dt.date.fromisoformat(end_date)

if _ed < _window_start:
    end_date   = _today.isoformat()
    start_date = (_today - _dt.timedelta(days=90)).isoformat()
    print(f"Date range outside 6-month sample window — reset to {start_date} / {end_date}")
elif _sd < _window_start:
    start_date = _window_start.isoformat()

print(f"Analysis period: {start_date} → {end_date}")

### Configuration

- **MIC**: Market Identifier Code for the exchange (e.g., XLON, XPAR, XNYS)
- **Ticker**: Exchange ticker as reported by the venue
- **Date range**: Analysis period — imbalance patterns are averaged across all trading days

Venues with well-populated closing auction imbalance data include:
`XLON` (LSE), `XPAR` (Euronext Paris), `XAMS` (Euronext Amsterdam),
`XETR` (XETRA), `XMIL` (Borsa Italiana), `XSWX` (SIX Swiss),
`XNAS` (NASDAQ), `XNYS` (NYSE).

### Step 1: Load Closing Auction Imbalance Data

**Reference price note:** Not all venues populate `REFERENCE_PRICE` in the imbalance feed.
Where it is absent we fall back to the auction mid-price
`(BEST_BID_PRICE + BEST_ASK_PRICE) / 2`, which the exchange reports when the book is not
yet fully crossed. If neither is available the value is carried forward from the previous
snapshot within the same day so that charts remain continuous.

In [ ]:
sql = f"""SELECT `TRADE_DATE`, `LOCAL_TIMESTAMP`, `MARKET_STATE`, `EXCHANGE_AUCTION_TYPE`, `IMBALANCE_SIDE`, COALESCE(NULLIF(`REFERENCE_PRICE`, 0), CASE WHEN `BEST_BID_PRICE` > 0 AND `BEST_ASK_PRICE` > 0 THEN (`BEST_BID_PRICE` + `BEST_ASK_PRICE`) / 2.0 END) AS `REFERENCE_PRICE` /* Effective reference price: use exchange-supplied value where available, */ /* fall back to mid of the auction BBO published in the same snapshot. */, `INDICATIVE_UNCROSSING_PRICE`, `AUCTION_ONLY_INDICATIVE_PRICE`, `PAIRED_QUANTITY`, `IMBALANCE_QUANTITY`, `IS_CROSSED` FROM {IMBALANCE_TBL} WHERE `TRADE_DATE` BETWEEN '{start_date}' AND '{end_date}' AND `MIC` = '{mic}' AND `EXCHANGE_TICKER` = '{ticker}' AND `MARKET_STATE` = 'CLOSING_AUCTION' ORDER BY `LOCAL_TIMESTAMP` NULLS LAST"""

df_raw = spark.sql(sql).toPandas()

print(f"Loaded {len(df_raw):,} closing auction snapshots")
print(f"Trading days: {df_raw['TRADE_DATE'].nunique()}")
print(f"\nImbalance side distribution:")
print(df_raw["IMBALANCE_SIDE"].value_counts().to_string())

### Step 2: Intraday Auction Dynamics

We aggregate all closing-auction snapshots by time-of-day (in 5-second bins)
to reveal the **typical evolution** of the auction across a full year of data.

Key metrics tracked:
- **Price dislocation (bps)**: `(indicative_uncrossing_price − reference_price) / reference_price × 10,000`
- **Imbalance quantity**: Excess buy or sell volume that cannot be matched
- **Paired quantity**: Volume successfully matched at the indicative price

In [ ]:
if df_raw.empty:
    print("No data found for the specified parameters.")
else:
    df = df_raw.copy()

    # Ensure LOCAL_TIMESTAMP is datetime
    df["LOCAL_TIMESTAMP"] = pd.to_datetime(df["LOCAL_TIMESTAMP"])

    # ── Reference price: three-tier fallback ─────────────────────────────────
    # Tier 1 (SQL): exchange-published REFERENCE_PRICE, or auction BBO mid when absent.
    # Tier 2 (Python): carry the last known value forward/backward within each day
    #   so that sparse publication (e.g., only the first snapshot) fills all rows.
    # Tier 3 (Python): if the venue publishes neither, anchor on the first valid
    #   INDICATIVE_UNCROSSING_PRICE of the day (dislocation then measures movement
    #   relative to the auction open rather than a pre-auction mid).
    df = df.sort_values(["TRADE_DATE", "LOCAL_TIMESTAMP"])

    # Zero → NaN so pandas fill functions treat them uniformly
    df["REFERENCE_PRICE"] = df["REFERENCE_PRICE"].replace(0, pd.NA)

    # Tier 2: ffill then bfill within each day
    df["REFERENCE_PRICE"] = (
        df.groupby("TRADE_DATE")["REFERENCE_PRICE"]
        .transform(lambda s: s.ffill().bfill())
    )

    # Tier 3: still missing → use first valid indicative price as per-day anchor
    _using_indicative_anchor = df["REFERENCE_PRICE"].isna().any()
    if _using_indicative_anchor:
        first_indicative = (
            df[df["INDICATIVE_UNCROSSING_PRICE"] > 0]
            .groupby("TRADE_DATE")["INDICATIVE_UNCROSSING_PRICE"]
            .first()
        )
        df["REFERENCE_PRICE"] = df["REFERENCE_PRICE"].fillna(
            df["TRADE_DATE"].map(first_indicative)
        )
        print("Note: reference price not available for this venue — "
              "using first indicative uncrossing price as per-day anchor "
              "(dislocation = movement relative to auction open).")

    # Keep only rows with a valid indicative price (required for all analyses).
    df = df[df["INDICATIVE_UNCROSSING_PRICE"].notna() & (df["INDICATIVE_UNCROSSING_PRICE"] > 0)]

    # Price dislocation from reference price (in basis points)
    df_with_ref = df[df["REFERENCE_PRICE"].notna() & (df["REFERENCE_PRICE"] > 0)].copy()
    df_with_ref["price_dislocation_bps"] = (
        (df_with_ref["INDICATIVE_UNCROSSING_PRICE"] - df_with_ref["REFERENCE_PRICE"])
        / df_with_ref["REFERENCE_PRICE"]
        * 10_000
    )

    # Time-of-day in seconds since midnight (for cross-date aggregation)
    for _d in [df, df_with_ref]:
        _d["tod_seconds"] = (
            _d["LOCAL_TIMESTAMP"].dt.hour * 3600
            + _d["LOCAL_TIMESTAMP"].dt.minute * 60
            + _d["LOCAL_TIMESTAMP"].dt.second
        )
        _d["tod_bin"] = (_d["tod_seconds"] // 5) * 5

    # Quantity aggregation over all rows with a valid indicative price
    avg_qty = df.groupby("tod_bin").agg(
        avg_imbalance_qty=("IMBALANCE_QUANTITY", "mean"),
        avg_paired_qty=("PAIRED_QUANTITY", "mean"),
        n_days=("TRADE_DATE", "nunique"),
    ).reset_index()
    avg_qty["time_label"] = pd.to_datetime(avg_qty["tod_bin"], unit="s", origin="1970-01-01").dt.strftime("%H:%M:%S")

    # Dislocation aggregation — only where reference price is available
    avg_dis = pd.DataFrame()
    if not df_with_ref.empty:
        avg_dis = df_with_ref.groupby("tod_bin").agg(
            avg_dislocation=("price_dislocation_bps", "mean"),
            p25_dislocation=("price_dislocation_bps", lambda x: x.quantile(0.25)),
            p75_dislocation=("price_dislocation_bps", lambda x: x.quantile(0.75)),
        ).reset_index()
        avg_dis["time_label"] = pd.to_datetime(avg_dis["tod_bin"], unit="s", origin="1970-01-01").dt.strftime("%H:%M:%S")

    print(f"Analysis covers {avg_qty['n_days'].max()} trading days")
    print(f"Auction time window: {avg_qty['time_label'].iloc[0]} — {avg_qty['time_label'].iloc[-1]}")

### Plot 1: Price Dislocation During the Closing Auction

Indicative price dislocation shows how far the auction's tentative execution price
strays from the reference price. As the auction proceeds and more orders arrive,
this dislocation typically narrows toward zero — signalling convergence to the final price.

Wide sustained dislocation indicates strong directional flow; tight convergence
suggests balanced order flow.

In [ ]:
if not df_raw.empty and not avg_dis.empty:
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.fill_between(
        avg_dis["tod_bin"],
        avg_dis["p25_dislocation"],
        avg_dis["p75_dislocation"],
        alpha=0.2,
        color="steelblue",
        label="25th–75th percentile",
    )
    ax.plot(avg_dis["tod_bin"], avg_dis["avg_dislocation"], color="steelblue", linewidth=2, label="Mean")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

    _ref_label = "Auction-Open Anchor" if _using_indicative_anchor else "Reference Price"
    ax.set_xlabel("Local time (HH:MM:SS)")
    ax.set_ylabel("Price dislocation (bps)")
    ax.set_title(
        f"Closing Auction: Indicative Price Dislocation from {_ref_label}\n"
        f"{ticker} on {mic}  |  {start_date} – {end_date}"
    )

    # X-axis: show every ~60 seconds
    tick_step = max(1, len(avg_dis) // 10)
    ax.set_xticks(avg_dis["tod_bin"].iloc[::tick_step])
    ax.set_xticklabels(avg_dis["time_label"].iloc[::tick_step], rotation=30, ha="right")

    ax.legend()
    plt.tight_layout()
    plt.show()
elif not df_raw.empty:
    print("Price dislocation chart skipped — reference price not available for this venue/ticker.")

### Plot 2: Imbalance and Paired Quantity Over Auction Time

As closing auction participants see the imbalance feed, they tend to send orders
to offset the surplus side. This typically leads to:
- Rising **paired quantity** as more volume is matched
- Falling **imbalance quantity** as the surplus is absorbed

In [ ]:
if not df_raw.empty and not avg_qty.empty:
    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.plot(avg_qty["tod_bin"], avg_qty["avg_paired_qty"], color="green", linewidth=2, label="Avg paired quantity")
    ax1.set_ylabel("Paired quantity (shares)", color="green")
    ax1.tick_params(axis="y", labelcolor="green")

    ax2 = ax1.twinx()
    ax2.plot(avg_qty["tod_bin"], avg_qty["avg_imbalance_qty"], color="red", linewidth=2, linestyle="--", label="Avg imbalance quantity")
    ax2.set_ylabel("Imbalance quantity (shares)", color="red")
    ax2.tick_params(axis="y", labelcolor="red")

    ax1.set_xlabel("Local time (HH:MM:SS)")
    ax1.set_title(
        f"Closing Auction: Order Matching vs. Imbalance Over Time\n"
        f"{ticker} on {mic}  |  {start_date} – {end_date}"
    )

    tick_step = max(1, len(avg_qty) // 10)
    ax1.set_xticks(avg_qty["tod_bin"].iloc[::tick_step])
    ax1.set_xticklabels(avg_qty["time_label"].iloc[::tick_step], rotation=30, ha="right")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    plt.tight_layout()
    plt.show()

### Step 3: Daily Imbalance Statistics

Beyond intraday patterns, we examine day-to-day variation in the closing auction.
The **end-of-auction imbalance** (last snapshot before the crossing) reveals
how much residual directional pressure existed on each day.

In [ ]:
if not df_raw.empty:
    # Last snapshot per day = closest to the actual close
    daily = (
        df.sort_values("LOCAL_TIMESTAMP")
        .groupby("TRADE_DATE")
        .last()
        .reset_index()
    )

    # Monthly summary
    daily["TRADE_DATE"] = pd.to_datetime(daily["TRADE_DATE"])
    daily["month"] = daily["TRADE_DATE"].dt.to_period("M").astype(str)
    monthly = daily.groupby("month").agg(
        avg_final_imbalance=("IMBALANCE_QUANTITY", "mean"),
        avg_paired_qty=("PAIRED_QUANTITY", "mean"),
        pct_bid_side=(
            "IMBALANCE_SIDE",
            lambda x: (x == "BID").mean() * 100
        ),
        pct_ask_side=(
            "IMBALANCE_SIDE",
            lambda x: (x == "ASK").mean() * 100
        ),
        trading_days=("TRADE_DATE", "nunique"),
    ).reset_index()

    print(f"\nMonthly closing auction summary for {ticker} on {mic}:")
    print(monthly.to_string(index=False, float_format="{:.0f}".format))

### Plot 3: Imbalance Side Distribution by Month

The balance between BID-side and ASK-side closing auction imbalances reflects
the net directional pressure participants bring to the close.
Persistent one-sided imbalance may indicate systematic buying (e.g., index flows)
or selling pressure.

In [ ]:
if not df_raw.empty and len(monthly) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))

    x = range(len(monthly))
    width = 0.4

    ax.bar([i - width / 2 for i in x], monthly["pct_bid_side"], width, label="BID-side imbalance (%)", color="steelblue")
    ax.bar([i + width / 2 for i in x], monthly["pct_ask_side"], width, label="ASK-side imbalance (%)", color="tomato")
    ax.axhline(50, color="black", linewidth=0.8, linestyle="--", alpha=0.4, label="50% balanced")

    ax.set_xlabel("Month")
    ax.set_ylabel("% of closing auctions")
    ax.set_title(
        f"Closing Auction Imbalance Side by Month\n"
        f"{ticker} on {mic}  |  {start_date} – {end_date}"
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels(monthly["month"], rotation=30, ha="right")
    ax.set_ylim(0, 100)
    ax.legend()
    plt.tight_layout()
    plt.show()

### Step 4: Single-Day Drill-Down

For a deeper look at price formation dynamics, we plot one representative day
to show how the indicative price, reference price, and imbalance evolve
minute-by-minute through the closing auction.

In [ ]:
if not df_raw.empty:
    # Pick the day with the most imbalance data as the representative day
    day_counts = df.groupby("TRADE_DATE")["LOCAL_TIMESTAMP"].count()
    representative_day = day_counts.idxmax()
    day_df = df[df["TRADE_DATE"] == representative_day].sort_values("LOCAL_TIMESTAMP")

    print(f"Representative day: {representative_day}  ({len(day_df)} snapshots)")

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Price evolution
    axes[0].plot(
        day_df["LOCAL_TIMESTAMP"], day_df["INDICATIVE_UNCROSSING_PRICE"],
        label="Indicative uncrossing price", color="steelblue", linewidth=1.5,
    )
    axes[0].plot(
        day_df["LOCAL_TIMESTAMP"], day_df["REFERENCE_PRICE"],
        label="Reference price", color="grey", linewidth=1.5, linestyle="--",
    )
    axes[0].set_ylabel("Price")
    axes[0].set_title(
        f"Closing Auction Price Formation — {representative_day}\n{ticker} on {mic}"
    )
    axes[0].legend()

    # Imbalance quantity and paired quantity
    axes[1].fill_between(
        day_df["LOCAL_TIMESTAMP"], 0, day_df["PAIRED_QUANTITY"],
        alpha=0.4, color="green", label="Paired quantity",
    )
    axes[1].plot(
        day_df["LOCAL_TIMESTAMP"], day_df["IMBALANCE_QUANTITY"],
        color="red", linewidth=1.5, label="Imbalance quantity",
    )
    axes[1].set_ylabel("Quantity (shares)")
    axes[1].set_xlabel("Local time")
    axes[1].legend()

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    plt.tight_layout()
    plt.show()